In [1]:
#!/usr/bin/env python3

# system
import sys 
sys.path.insert(0, '../../../')

# utils
import numpy as np
import torch  
import logging
from tqdm import tqdm 
from chessrl.utils.load_config import load_config
from chessrl.utils.fen_parsing import parse_fen_cached
from typing import List, Tuple, Dict


import os
config_path = os.path.join('./', 'config.json')
config = load_config(config_path)
logging.basicConfig(level=config['log_level'], format = '%(asctime)s - %(levelname)s - %(message)s')
logger = logging.getLogger(__name__)

# chess
from chessrl import Env, SyzygyDefender
from chessrl import chess_py as cp
from chessrl.algorithms.actor_critic.policy import Policy
from chessrl.algorithms.actor_critic.ac import ActorCritic
from chessrl.utils.move_idx import build_move_mappings

move_to_idx, idx_to_move = build_move_mappings()

2025-08-30 11:51:22,697 - INFO - Loading config file...
2025-08-30 11:51:23,160 - INFO - Loading config file...
2025-08-30 11:51:23,166 - INFO - Loading config file...


In [3]:
agent = ActorCritic()

In [3]:
fen = "1K6/8/3k4/8/8/8/5R2/8 w - - 0 1"

In [5]:
features = agent.obtain_features(fen)

In [6]:
features

[2, 1, 3, 1, 1]

In [7]:
[features]

[[2, 1, 3, 1, 1]]

# Policy

In [2]:
policy = Policy()

In [4]:
env = Env.from_fen(fen)

In [11]:
mv = env.state().legal_moves(cp.Color.WHITE)[0]

In [12]:
cp.Move.to_uci(mv)

'f2f1'

In [39]:
legal_moves_idx = [2,3,4]

In [40]:
from chessrl.utils.fen_parsing import parse_fen
state_tensor = parse_fen(fen).unsqueeze(0).permute(0,3,1,2)

In [42]:
logits = policy.forward(state_tensor)
logits = logits.squeeze(0) # [4096]
logits

tensor([ 0.0245, -0.3707,  0.7702,  ...,  0.0603,  0.2551, -0.0172],
       grad_fn=<SqueezeBackward1>)

In [43]:
logits_actions_dict = {k: logits[k] for k in range(4096)}
logits_actions_dict

{0: tensor(0.0245, grad_fn=<SelectBackward0>),
 1: tensor(-0.3707, grad_fn=<SelectBackward0>),
 2: tensor(0.7702, grad_fn=<SelectBackward0>),
 3: tensor(-0.3072, grad_fn=<SelectBackward0>),
 4: tensor(0.0914, grad_fn=<SelectBackward0>),
 5: tensor(0.1016, grad_fn=<SelectBackward0>),
 6: tensor(-0.1000, grad_fn=<SelectBackward0>),
 7: tensor(0.8806, grad_fn=<SelectBackward0>),
 8: tensor(0.2105, grad_fn=<SelectBackward0>),
 9: tensor(0.5808, grad_fn=<SelectBackward0>),
 10: tensor(0.0786, grad_fn=<SelectBackward0>),
 11: tensor(0.2440, grad_fn=<SelectBackward0>),
 12: tensor(-0.3705, grad_fn=<SelectBackward0>),
 13: tensor(-0.3601, grad_fn=<SelectBackward0>),
 14: tensor(0.5125, grad_fn=<SelectBackward0>),
 15: tensor(0.0805, grad_fn=<SelectBackward0>),
 16: tensor(0.5962, grad_fn=<SelectBackward0>),
 17: tensor(0.3855, grad_fn=<SelectBackward0>),
 18: tensor(-0.3750, grad_fn=<SelectBackward0>),
 19: tensor(0.3799, grad_fn=<SelectBackward0>),
 20: tensor(-0.1138, grad_fn=<SelectBackward

In [44]:
legal_logits = {k: logits_actions_dict[k] for k in legal_moves_idx}
legal_logits

{2: tensor(0.7702, grad_fn=<SelectBackward0>),
 3: tensor(-0.3072, grad_fn=<SelectBackward0>),
 4: tensor(0.0914, grad_fn=<SelectBackward0>)}

In [45]:
idxs = list(legal_logits.keys())
idxs

[2, 3, 4]

In [46]:
values = torch.stack([legal_logits[k] for k in idxs])
values

tensor([ 0.7702, -0.3072,  0.0914], grad_fn=<StackBackward0>)

In [47]:
legal_probs = torch.softmax(values, dim=0)
legal_probs

tensor([0.5412, 0.1843, 0.2745], grad_fn=<SoftmaxBackward0>)

In [48]:
legal_probs_dict = {k: v for k, v in zip(idxs, legal_probs)}
legal_probs_dict

{2: tensor(0.5412, grad_fn=<UnbindBackward0>),
 3: tensor(0.1843, grad_fn=<UnbindBackward0>),
 4: tensor(0.2745, grad_fn=<UnbindBackward0>)}

In [49]:
action_idx = max(legal_probs_dict, key=legal_probs_dict.get)
action_idx

2

In [59]:
selected_idx = torch.multinomial(legal_probs, 1).item() # samples action on reduced action space (only legal moves)
action_idx_sampled = idxs[selected_idx]
action_idx_sampled

2

In [63]:
log_legal_prob = torch.log(legal_probs_dict[action_idx])
log_legal_prob

tensor(-0.6139, grad_fn=<LogBackward0>)

In [70]:
env.to_fen()

'1K6/8/3k4/8/8/8/5R2/8 w - - 0 1'